In [1]:
# =====================================================================
# Build the GenImage project subset (Optimized Multi-Threaded Version)
# =====================================================================

import os, io, re, csv, gc, random, zipfile, bisect
import concurrent.futures
import threading
from collections import defaultdict
from PIL import Image

# ---- what this run processes ----
ONLY_GENS = ["ADM", "BigGAN","GLIDE","Midjourney","SD14","SD15","VQDM","Wukong"]

SEED = 42
INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working"
IMG_SIZE = 256
JPEG_QUALITY = 95
N_TRAIN_PER_CLASS = 6250   
N_EVAL_PER_CLASS = 2000    
MAX_PIXELS = 100_000_000   

Image.MAX_IMAGE_PIXELS = None   

GENERATORS = {
    "ADM":        ["adm"],
    "BigGAN":     ["biggan"],
    "GLIDE":      ["glide"],
    "Midjourney": ["midjourney"],
    "SD14":       ["v1-4", "v-1-4", "v14", "sdv4"],
    "SD15":       ["v1-5", "v-1-5", "v15", "sdv5"],
    "VQDM":       ["vqdm"],
    "Wukong":     ["wukong"],
}

GEN_ID = {"Real": 0, "ADM": 1, "BigGAN": 2, "GLIDE": 3, "Midjourney": 4,
          "SD14": 5, "SD15": 6, "VQDM": 7, "Wukong": 8}

IMG_EXT = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
SKIP_EXT = (".txt", ".md", ".json", ".csv", ".py", ".ipynb", ".html", ".part")
FIELDS = ["file", "split", "label", "label_name", "generator", "class9", "source_path"]

# Thread lock for safe concurrent I/O on archives
io_lock = threading.Lock()

def find_dataset_roots():
    roots = []
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        depth = dirpath[len(INPUT_ROOT):].count(os.sep)
        name = os.path.basename(dirpath).lower()
        if "genimage" in name and "validation" not in name:
            roots.append(dirpath)
            dirnames[:] = []          
            continue
        if depth >= 3:
            dirnames[:] = []
    return roots

def match_generator(root):
    name = os.path.basename(root).lower().replace("_", "-")
    for gen, keys in GENERATORS.items():
        if any(k in name for k in keys):
            return gen
    return None

def find_outputs(filename):
    found = []
    p = os.path.join(WORK_DIR, filename)
    if os.path.exists(p): found.append(p)
    for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
        if dirpath[len(INPUT_ROOT):].count(os.sep) >= 3:
            dirnames[:] = []
        if filename in filenames:
            found.append(os.path.join(dirpath, filename))
    return found

class ConcatFile(io.RawIOBase):
    def __init__(self, paths):
        self.files = [open(p, "rb") for p in paths]
        self.sizes = [os.path.getsize(p) for p in paths]
        self.starts = [0]
        for s in self.sizes[:-1]: self.starts.append(self.starts[-1] + s)
        self.total = sum(self.sizes)
        self.pos = 0
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, offset, whence=0):
        if whence == 0: self.pos = offset
        elif whence == 1: self.pos += offset
        else: self.pos = self.total + offset
        return self.pos
    def readinto(self, b):
        n, got = len(b), 0
        while got < n and self.pos < self.total:
            i = bisect.bisect_right(self.starts, self.pos) - 1
            local = self.pos - self.starts[i]
            f = self.files[i]
            f.seek(local)
            chunk = f.read(min(n - got, self.sizes[i] - local))
            if not chunk: break
            b[got:got + len(chunk)] = chunk
            got += len(chunk)
            self.pos += len(chunk)
        return got
    def close(self):
        for f in self.files: f.close()
        super().close()

def parse_path(path):
    segs = [s.lower() for s in re.split(r"[\\/]", path)]
    split = "train" if "train" in segs else ("val" if "val" in segs else None)
    cls = "fake" if "ai" in segs else ("real" if "nature" in segs else None)
    return split, cls

def collect_pools(roots):
    images, zips, parts = [], [], []
    for r in roots:
        for dp, _, fns in os.walk(r):
            for fn in fns:
                low = fn.lower()
                if low.startswith("genimage_"): continue
                p = os.path.join(dp, fn)
                if low.endswith(IMG_EXT): images.append(p)
                elif low.endswith(".zip"): zips.append(p)
                elif not low.endswith(SKIP_EXT): parts.append(p)

    pools = defaultdict(list)
    if images:                                  
        for p in images:
            s, c = parse_path(p)
            if s and c: pools[(s, c)].append(p)
        return pools, "folders", []

    if zips:
        archives = [zipfile.ZipFile(z) for z in sorted(zips)]
        mode = "zip"
    elif parts:                                  
        parts.sort(key=lambda p: os.path.basename(p))
        raw = io.BufferedReader(ConcatFile(parts), buffer_size=1 << 20)
        archives = [zipfile.ZipFile(raw)]
        mode = f"split zip ({len(parts)} parts)"
    else:
        return pools, "nothing found", []

    for zf in archives:
        for m in zf.namelist():
            if m.lower().endswith(IMG_EXT):
                s, c = parse_path(m)
                if s and c: pools[(s, c)].append((zf, m))
    return pools, mode, archives

def ref_name(ref):
    return ref if isinstance(ref, str) else ref[1]

def load_used_real():
    used, seen = set(), []
    for gen in GENERATORS:
        for p in find_outputs(f"genimage_{gen}.csv"):
            with open(p, newline="") as f:
                for r in csv.DictReader(f):
                    if r["label_name"] == "real":
                        used.add(os.path.basename(r["source_path"]).lower())
            seen.append(gen)
            break
    return used, seen

def process_single_image(args):
    """Worker function for parallel processing."""
    ref, src_split, cls, taken, need, gen = args
    try:
        # NO LOCK HERE - Read loose files concurrently across all CPUs
        if isinstance(ref, str):
            with open(ref, "rb") as f: 
                img_bytes = f.read()
        
        # ONLY lock if reading from a shared zip archive (e.g. Midjourney)
        else:
            with io_lock:
                zf, m = ref
                img_bytes = zf.read(m)
                
        img = Image.open(io.BytesIO(img_bytes))
        w, h = img.size
        if w * h > MAX_PIXELS: return None
        img.load()

        # Heavy CPU ops outside the lock
        img = img.convert("RGB")
        s = IMG_SIZE / min(w, h)
        img = img.resize((max(IMG_SIZE, round(w * s)), max(IMG_SIZE, round(h * s))), Image.BICUBIC)
        w, h = img.size
        left, top = (w - IMG_SIZE) // 2, (h - IMG_SIZE) // 2
        img = img.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))

        split = "train" if src_split == "train" else ("val" if taken < need // 2 else "test")
        fname = f"{split}/{gen}_{cls}_{src_split}_{taken:05d}.jpg"
        
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=JPEG_QUALITY)
        
        row = {
            "file": fname, "split": split, "label": 0 if cls == "real" else 1,
            "label_name": cls, "generator": gen,
            "class9": 0 if cls == "real" else GEN_ID[gen], "source_path": ref_name(ref),
        }
        return fname, buf.getvalue(), row
    except Exception:
        return None

def build_generator(gen, roots, used_real):
    out_zip = os.path.join(WORK_DIR, f"genimage_{gen}.zip")
    out_csv = os.path.join(WORK_DIR, f"genimage_{gen}.csv")
    tmp_zip = out_zip + ".part"                  

    rng = random.Random(f"{SEED}-{gen}")         
    pools, mode, archives = collect_pools(roots)
    print(f"\n[{gen}] read as {mode}; available: "
          + ", ".join(f"{k[0]}/{k[1]}={len(v)}" for k, v in sorted(pools.items())))

    for key in pools:                            
        pools[key].sort(key=ref_name)
        rng.shuffle(pools[key])

    plan = [("train", "real", N_TRAIN_PER_CLASS), ("train", "fake", N_TRAIN_PER_CLASS),
            ("val", "real", N_EVAL_PER_CLASS), ("val", "fake", N_EVAL_PER_CLASS)]
    rows = []
    
    max_workers = os.cpu_count() or 4

    try:
        with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_STORED) as out:
            for src_split, cls, need in plan:
                tasks = []
                taken = skipped = 0
                
                for ref in pools[(src_split, cls)]:
                    if taken >= need: break
                    base = os.path.basename(ref_name(ref)).lower()
                    
                    if cls == "real" and base in used_real:
                        skipped += 1
                        continue
                        
                    if cls == "real":
                        used_real.add(base)
                        
                    tasks.append((ref, src_split, cls, taken, need, gen))
                    taken += 1

                processed_count = 0
                with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
                    futures = {executor.submit(process_single_image, task): task for task in tasks}
                    
                    for future in concurrent.futures.as_completed(futures):
                        result = future.result()
                        if result:
                            fname, img_bytes, row_data = result
                            out.writestr(fname, img_bytes)
                            rows.append(row_data)
                            processed_count += 1
                            if processed_count % 2000 == 0:
                                print(f"    {src_split}/{cls}: {processed_count}/{need}")
                        else:
                            skipped += 1

                status = "OK" if processed_count == need else f"WARNING: only {processed_count}/{need}"
                print(f"  {src_split}/{cls}: {status} (skipped {skipped})")
    finally:
        for zf in archives:                      
            zf.close()
        del pools, archives
        gc.collect()

    os.replace(tmp_zip, out_zip)
    with open(out_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows(rows)
    counts = {s: sum(r["split"] == s for r in rows) for s in ["train", "val", "test"]}
    print(f"  saved {out_zip} ({os.path.getsize(out_zip) / 1e9:.2f} GB) {counts}")

def main():
    unknown = [g for g in ONLY_GENS if g not in GENERATORS]
    if unknown: raise SystemExit(f"Unknown generator names in ONLY_GENS: {unknown}")

    by_gen = defaultdict(list)
    for r in find_dataset_roots():
        g = match_generator(r)
        if g in ONLY_GENS: by_gen[g].append(r)
        
    print("Datasets for this run:")
    for g in ONLY_GENS: print(f"  {g:11s} -> {by_gen.get(g, 'MISSING')}")
        
    missing = [g for g in ONLY_GENS if g not in by_gen]
    if missing: raise SystemExit(f"Add these datasets via 'Add Input' first: {missing}")

    used_real, done = load_used_real()
    print(f"Earlier runs found: {done or 'none'} ({len(used_real)} real images already used)")
    if not done: print("  (if you already ran other generators, attach their outputs via 'Add Input')")

    for gen in ONLY_GENS:
        if gen in done:
            print(f"\n[{gen}] already built in an earlier run - skipping")
            continue
        build_generator(gen, by_gen[gen], used_real)

if ONLY_GENS:
    main()

Datasets for this run:
  ADM         -> ['/kaggle/input/datasets/vtphatt2/genimage-adm']
  BigGAN      -> ['/kaggle/input/datasets/vtphatt2/genimage-biggan']
  GLIDE       -> ['/kaggle/input/datasets/vtphatt2/genimage-glide']
  Midjourney  -> ['/kaggle/input/datasets/vtphatt2/genimage-midjourney-part-1', '/kaggle/input/datasets/vtphatt2/genimage-midjourney-part-3', '/kaggle/input/datasets/vtphatt2/genimage-midjourney-part-2']
  SD14        -> ['/kaggle/input/datasets/vtphatt2/genimage-stable-diffusion-v1-4']
  SD15        -> ['/kaggle/input/datasets/vtphatt2/genimage-stable-diffusion-v1-5']
  VQDM        -> ['/kaggle/input/datasets/vtphatt2/genimage-vqdm']
  Wukong      -> ['/kaggle/input/datasets/vtphatt2/genimage-wukong']
Earlier runs found: none (0 real images already used)
  (if you already ran other generators, attach their outputs via 'Add Input')

[ADM] read as folders; available: train/fake=162000, train/real=157453, val/fake=6000, val/real=6000
    train/real: 2000/6250
    tr

In [4]:
# =====================================================================
# MERGE - run once after all 8 generators are built (set ONLY_GENS = []).
# Attach every run's output (genimage_<GEN>.zip + .csv) via "Add Input"
# (no GenImage datasets needed). Streams files, so RAM use stays small.
#
# Output in /kaggle/working:
#   genimage_subset.zip   train/ val/ test/ + train.csv, val.csv, test.csv
#   train.csv, val.csv, test.csv
# =====================================================================

import shutil
from collections import Counter

def merge():
    OUT_ZIP = os.path.join(WORK_DIR, "genimage_subset.zip")

    sources = {}
    for gen in GENERATORS:
        z, c = find_outputs(f"genimage_{gen}.zip"), find_outputs(f"genimage_{gen}.csv")
        if z and c:
            sources[gen] = (z[0], c[0])
    missing = [g for g in GENERATORS if g not in sources]
    if missing:
        raise SystemExit(f"Missing per-generator outputs (build or attach them first): {missing}")

    rows = []
    with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_STORED) as out:
        for gen, (zpath, cpath) in sources.items():
            print(f"adding {gen} from {zpath}")
            with zipfile.ZipFile(zpath) as zin:
                for info in zin.infolist():
                    with zin.open(info) as src, out.open(info.filename, "w") as dst:
                        shutil.copyfileobj(src, dst, 1 << 20)
            with open(cpath, newline="") as f:
                rows.extend(csv.DictReader(f))

        for split in ["train", "val", "test"]:
            sio = io.StringIO()
            w = csv.DictWriter(sio, fieldnames=FIELDS)
            w.writeheader()
            w.writerows(r for r in rows if r["split"] == split)
            out.writestr(f"{split}.csv", sio.getvalue())
            with open(os.path.join(WORK_DIR, f"{split}.csv"), "w", newline="") as f:
                f.write(sio.getvalue())

    # summary
    print("\n===== SUMMARY =====")
    for split in ["train", "val", "test"]:
        c = Counter((r["generator"], r["label_name"]) for r in rows if r["split"] == split)
        print(f"{split}: {sum(c.values())} images")
    reals = Counter(os.path.basename(r["source_path"]).lower() for r in rows if r["label_name"] == "real")
    dups = sum(1 for n in reals.values() if n > 1)
    print(f"duplicate real images across generators: {dups}"
          + ("  (attach earlier outputs when building to avoid this)" if dups else ""))
    print(f"Saved: {OUT_ZIP}  ({os.path.getsize(OUT_ZIP) / 1e9:.2f} GB)")
ONLY_GENS = []

if not ONLY_GENS:
    merge()
else:
    print("Build run - merge skipped (set ONLY_GENS = [] to merge).")


adding ADM from /kaggle/working/genimage_ADM.zip
adding BigGAN from /kaggle/working/genimage_BigGAN.zip
adding GLIDE from /kaggle/working/genimage_GLIDE.zip
adding Midjourney from /kaggle/working/genimage_Midjourney.zip
adding SD14 from /kaggle/working/genimage_SD14.zip
adding SD15 from /kaggle/working/genimage_SD15.zip
adding VQDM from /kaggle/working/genimage_VQDM.zip
adding Wukong from /kaggle/working/genimage_Wukong.zip

===== SUMMARY =====
train: 100000 images
val: 16000 images
test: 16000 images
duplicate real images across generators: 0
Saved: /kaggle/working/genimage_subset.zip  (3.91 GB)


In [ ]:
import os
from IPython.display import FileLink

filename = 'genimage_subset.zip' 

FileLink(filename)

In [8]:
import json
import os

# Create the required metadata file
meta = {
  "title": "GenImage Final Subset",
  "id": "nafeesamahek/genimage-final-subset",
  "licenses": [{"name": "CC0-1.0"}]
}

with open("/kaggle/working/dataset-metadata.json", "w") as f:
    json.dump(meta, f)

# Upload directly to Kaggle Datasets
!kaggle datasets create -p /kaggle/working/

Starting upload for file val.csv
100%|██████████████████████████████████████| 2.37M/2.37M [00:01<00:00, 2.12MB/s]
Upload successful: val.csv (2MB)
Starting upload for file genimage_VQDM.csv
100%|██████████████████████████████████████| 2.38M/2.38M [00:01<00:00, 2.03MB/s]
Upload successful: genimage_VQDM.csv (2MB)
Starting upload for file genimage_SD14.zip
100%|████████████████████████████████████████| 507M/507M [00:13<00:00, 39.5MB/s]
Upload successful: genimage_SD14.zip (507MB)
Starting upload for file genimage_subset.zip
100%|██████████████████████████████████████| 3.64G/3.64G [01:31<00:00, 42.8MB/s]
Upload successful: genimage_subset.zip (4GB)
Starting upload for file genimage_GLIDE.csv
100%|██████████████████████████████████████| 2.56M/2.56M [00:01<00:00, 2.27MB/s]
Upload successful: genimage_GLIDE.csv (3MB)
Starting upload for file genimage_BigGAN.csv
100%|██████████████████████████████████████| 2.63M/2.63M [00:01<00:00, 2.29MB/s]
Upload successful: genimage_BigGAN.csv (3MB)
Starti